# ✈️ Airport Regime
## Regime-Adaptive Prediction of Near-Term Flight Disruptions

### Core question

> **Does an airport operate under different latent conditions, and does recognizing those conditions improve near-term disruption prediction?**

**Flight events → airport-hour state → exact temporal lags → tabular ML → latent regimes → regime-aware ML**

**Data:** 3M U.S. flight records, 2019–2023 sample  
**Target:** unusually high delay rate in the next operating hour  
**Models:** seasonal prior, Logistic Regression, LightGBM, CatBoost  
**Regime layer:** Gaussian-mixture latent states + empirical transitions  
**Validation:** chronological; final period remains untouched

> This is an applied ML experiment, not a claim of a new algorithm.

In [2]:
# ============================================================
# CELL 1 — ENVIRONMENT & REPRODUCIBILITY
# Keep startup lightweight. No runtime package installation.
# ============================================================

import os
import gc
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    precision_recall_curve,
)
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier
import shap

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_PATH = (
    "/kaggle/input/datasets/patrickzel/"
    "flight-delay-and-cancellation-dataset-2019-2023/"
    "flights_sample_3m.csv"
)

TRAIN_END = pd.Timestamp("2022-01-01")
VAL_END = pd.Timestamp("2023-01-01")

COLORS = {
    "blue": "#2563EB",
    "cyan": "#06B6D4",
    "purple": "#7C3AED",
    "orange": "#F97316",
    "green": "#10B981",
    "red": "#EF4444",
    "gray": "#64748B",
    "navy": "#0F172A",
}

try:
    import torch
    GPU_COUNT = torch.cuda.device_count()
    GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
except Exception:
    GPU_COUNT = 0
    GPU_NAMES = []

print("✈️ Airport Regime")
print("=" * 60)
print(f"Dataset found : {os.path.exists(DATA_PATH)}")
print(f"Train end     : {TRAIN_END.date()}")
print(f"Validation end: {VAL_END.date()}")
print(f"GPUs visible  : {GPU_COUNT}")
for i, name in enumerate(GPU_NAMES):
    print(f"  GPU {i}: {name}")

✈️ Airport Regime
Dataset found : True
Train end     : 2022-01-01
Validation end: 2023-01-01
GPUs visible  : 1
  GPU 0: Tesla P100-PCIE-16GB


In [3]:
# ============================================================
# CELL 2 — LOAD + PARSE RAW FLIGHT EVENTS
# We inspect the header first so the notebook adapts to the
# actual Kaggle file instead of assuming fragile column names.
# ============================================================

available_cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()

PREFERRED_COLS = [
    "FL_DATE", "AIRLINE_CODE", "ORIGIN", "DEST",
    "CRS_DEP_TIME", "DEP_TIME", "DEP_DELAY",
    "CRS_ARR_TIME", "ARR_TIME", "ARR_DELAY",
    "CANCELLED", "DIVERTED",
    "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS", "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT",
]

required = [
    "FL_DATE", "ORIGIN", "DEST",
    "CRS_DEP_TIME", "DEP_DELAY", "CANCELLED"
]

missing_required = [c for c in required if c not in available_cols]
if missing_required:
    raise ValueError(f"Required columns missing from dataset: {missing_required}")

USE_COLS = [c for c in PREFERRED_COLS if c in available_cols]
df = pd.read_csv(DATA_PATH, usecols=USE_COLS, low_memory=False)

df["FL_DATE"] = pd.to_datetime(df["FL_DATE"], errors="coerce")

def hhmm_to_minutes(series):
    x = pd.to_numeric(series, errors="coerce")
    hour = np.floor(x / 100)
    minute = x % 100
    valid = hour.between(0, 23) & minute.between(0, 59)
    return (hour * 60 + minute).where(valid)

for source, target in [
    ("CRS_DEP_TIME", "CRS_DEP_MIN"),
    ("DEP_TIME", "DEP_MIN"),
    ("CRS_ARR_TIME", "CRS_ARR_MIN"),
    ("ARR_TIME", "ARR_MIN"),
]:
    if source in df.columns:
        df[target] = hhmm_to_minutes(df[source])

df["IS_DELAYED"] = (
    pd.to_numeric(df["DEP_DELAY"], errors="coerce")
    .fillna(0).ge(15).astype("int8")
)
df["IS_CANCELLED"] = (
    df["CANCELLED"].fillna(0).eq(1).astype("int8")
)
df["IS_DIVERTED"] = (
    df["DIVERTED"].fillna(0).eq(1).astype("int8")
)

print("Raw data loaded.")
print("-" * 60)
print(f"Rows        : {len(df):,}")
print(f"Columns     : {df.shape[1]}")
print(f"Coverage    : {df['FL_DATE'].min().date()} → {df['FL_DATE'].max().date()}")
print(f"Airports    : {df['ORIGIN'].nunique():,}")
print(f"Airlines    : {df['AIRLINE_CODE'].nunique():,}")
print(f"Delay rate  : {df['IS_DELAYED'].mean():.1%}")
print(f"Cancel rate : {df['IS_CANCELLED'].mean():.1%}")

Raw data loaded.
------------------------------------------------------------
Rows        : 3,000,000
Columns     : 24
Coverage    : 2019-01-01 → 2023-08-31
Airports    : 380
Airlines    : 18
Delay rate  : 17.7%
Cancel rate : 2.6%


In [9]:
# ============================================================
# CELL 3 — TWO VIEWS THAT JUSTIFY THE MODELING SCOPE
# 1) A day × hour delay fingerprint.
# 2) An airport bubble chart used to select a focused universe.
# Airport selection uses training data only.
# ============================================================

train_rows = df[df["FL_DATE"] < TRAIN_END].copy()
train_rows["dow"] = train_rows["FL_DATE"].dt.dayofweek
train_rows["dep_hour"] = (train_rows["CRS_DEP_MIN"] // 60).astype("float")

fingerprint = (
    train_rows.dropna(subset=["dep_hour"])
    .groupby(["dow", "dep_hour"])["IS_DELAYED"]
    .mean().mul(100)
    .unstack()
    .reindex(index=range(7), columns=range(24))
)
fingerprint.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

airport_stats = (
    train_rows.groupby("ORIGIN")
    .agg(
        flights=("ORIGIN", "size"),
        delay_rate=("IS_DELAYED", "mean"),
        avg_delay=("DEP_DELAY", "mean"),
    )
)
airport_stats = airport_stats[airport_stats["flights"] >= 20_000].copy()
airport_stats["delay_rate"] *= 100

airport_stats["selection_score"] = (
    0.5 * airport_stats["flights"].rank(pct=True)
    + 0.5 * airport_stats["delay_rate"].rank(pct=True)
)

MODELING_AIRPORTS = (
    airport_stats.sort_values("selection_score", ascending=False)
    .head(8).index.astype(str).tolist()
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Operational fingerprint · when do delays rise?",
        "Airport operating envelope · traffic × delay",
    ),
    column_widths=[0.52, 0.48],
)

fig.add_trace(
    go.Heatmap(
        z=fingerprint.values,
        x=list(range(24)),
        y=list(fingerprint.index),
        colorscale="Turbo",
        text=np.round(fingerprint.values, 1),
        texttemplate="%{text}",
        hovertemplate=(
            "Day: %{y}<br>Hour: %{x}:00<br>"
            "Delay rate: %{z:.1f}%<extra></extra>"
        ),
        colorbar=dict(title="Delay %", x=0.47),
    ),
    row=1, col=1,
)

b = airport_stats.reset_index()
fig.add_trace(
    go.Scatter(
        x=b["flights"],
        y=b["delay_rate"],
        mode="markers+text",
        text=b["ORIGIN"],
        textposition="top center",
        marker=dict(
            size=np.clip(b["avg_delay"].fillna(5) * 1.6, 7, 34),
            color=b["delay_rate"],
            colorscale="Turbo",
            showscale=False,
            opacity=0.88,
            line=dict(width=1, color="white"),
        ),
        customdata=np.c_[b["flights"], b["avg_delay"]],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Flights: %{customdata[0]:,.0f}<br>"
            "Delay rate: %{y:.1f}%<br>"
            "Mean delay: %{customdata[1]:.1f} min<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_xaxes(title_text="Scheduled departure hour", row=1, col=1)
fig.update_yaxes(title_text="Day of week", row=1, col=1)
fig.update_xaxes(title_text="Training-period flight volume", type="log", row=1, col=2)
fig.update_yaxes(title_text="Departure delay rate (%)", row=1, col=2)
fig.update_layout(
    template="plotly_white",
    height=520,
    title="The airport is not one stationary process",
    title_x=0.5,
)
fig.show()

print("Modeling airports:", ", ".join(MODELING_AIRPORTS))

Modeling airports: DEN, DFW, ORD, LAS, MCO, PHX, EWR, LAX


### Scope decision

We keep eight high-volume airports. The goal is not to model every U.S. airport; it is to create a clean enough operational panel to test whether **state changes** matter.

In [10]:
# ============================================================
# CELL 4 — FLIGHT EVENTS → AIRPORT × HOUR OPERATIONAL PANEL
# Local workload is based on scheduled departure hour.
# Inbound pressure uses actual arrival time when available,
# because that is an observed network signal by the time we
# forecast the next hour.
# ============================================================

model_df = df[
    df["ORIGIN"].astype(str).isin(MODELING_AIRPORTS)
    | df["DEST"].astype(str).isin(MODELING_AIRPORTS)
].copy()

# ---- Local departure operations ------------------------------
dep = model_df[
    model_df["ORIGIN"].astype(str).isin(MODELING_AIRPORTS)
].copy()

dep["timestamp"] = (
    dep["FL_DATE"] + pd.to_timedelta(dep["CRS_DEP_MIN"], unit="m")
)
dep["operated"] = dep["IS_CANCELLED"].eq(0)
dep["delayed_15"] = dep["DEP_DELAY"].fillna(0).ge(15)

dep_hourly = (
    dep.groupby(["ORIGIN", "timestamp"], as_index=False)
    .agg(
        scheduled_departures=("ORIGIN", "size"),
        operated_departures=("operated", "sum"),
        delayed_departures=("delayed_15", "sum"),
        avg_departure_delay=("DEP_DELAY", "mean"),
        cancellation_rate=("IS_CANCELLED", "mean"),
    )
    .rename(columns={"ORIGIN": "airport"})
)

dep_hourly["delay_rate"] = (
    dep_hourly["delayed_departures"]
    / dep_hourly["operated_departures"].replace(0, np.nan)
)

# ---- Inbound pressure ----------------------------------------
if "ARR_TIME" in model_df.columns:
    arrival_time_col = "ARR_MIN"
elif "CRS_ARR_TIME" in model_df.columns:
    arrival_time_col = "CRS_ARR_MIN"
else:
    arrival_time_col = None

if arrival_time_col is not None:
    arr = model_df[
        model_df["DEST"].astype(str).isin(MODELING_AIRPORTS)
        & model_df[arrival_time_col].notna()
    ].copy()

    arr["timestamp"] = (
        arr["FL_DATE"]
        + pd.to_timedelta(arr[arrival_time_col], unit="m")
    )
    arr["operated"] = arr["IS_CANCELLED"].eq(0)

    arr_hourly = (
        arr.groupby(["DEST", "timestamp"], as_index=False)
        .agg(
            inbound_flights=("DEST", "size"),
            inbound_operated=("operated", "sum"),
            inbound_avg_delay=("ARR_DELAY", "mean"),
        )
        .rename(columns={"DEST": "airport"})
    )
else:
    arr_hourly = pd.DataFrame(
        columns=[
            "airport", "timestamp",
            "inbound_flights", "inbound_operated",
            "inbound_avg_delay",
        ]
    )

panel = (
    dep_hourly
    .merge(arr_hourly, on=["airport", "timestamp"], how="left")
    .sort_values(["airport", "timestamp"])
    .reset_index(drop=True)
)

panel["hour"] = panel["timestamp"].dt.hour
panel["day_of_week"] = panel["timestamp"].dt.dayofweek
panel["month"] = panel["timestamp"].dt.month
panel["is_weekend"] = panel["day_of_week"].ge(5).astype("int8")
panel["is_peak"] = panel["hour"].isin(
    [7, 8, 9, 16, 17, 18, 19]
).astype("int8")

print(f"Airport-hour observations: {len(panel):,}")
print(f"Airports: {panel['airport'].nunique()}")
display(panel.head())

Airport-hour observations: 646,481
Airports: 8


,airport,timestamp,scheduled_departures,operated_departures,delayed_departures,avg_departure_delay,cancellation_rate,delay_rate,inbound_flights,inbound_operated,inbound_avg_delay,hour,day_of_week,month,is_weekend,is_peak
0,DEN,2019-01-01 05:33:00,1,1,0,5.000,0.000,0.000,NaN,NaN,NaN,5,1,1,0,0
1,DEN,2019-01-01 05:50:00,1,1,0,1.000,0.000,0.000,NaN,NaN,NaN,5,1,1,0,0
2,DEN,2019-01-01 07:55:00,2,2,0,2.000,0.000,0.000,NaN,NaN,NaN,7,1,1,0,1
3,DEN,2019-01-01 07:57:00,1,1,1,21.000,0.000,1.000,NaN,NaN,NaN,7,1,1,0,1
4,DEN,2019-01-01 08:15:00,1,1,0,2.000,0.000,0.000,NaN,NaN,NaN,8,1,1,0,1


In [20]:
# ============================================================
# CELL 5 — BUILD AIRPORT × HOUR OPERATIONAL PANEL
#
# Convert flight-level events into a true hourly panel.
# Every timestamp is floored to the hour:
#
#     08:17 → 08:00
#     08:42 → 08:00
#     09:03 → 09:00
#
# This is essential because our target looks one/two HOURS ahead.
# ============================================================

# ------------------------------------------------------------
# 1. Keep flights touching selected airports
# ------------------------------------------------------------

model_df = df[
    df["ORIGIN"].astype(str).isin(MODELING_AIRPORTS)
    |
    df["DEST"].astype(str).isin(MODELING_AIRPORTS)
].copy()

print(f"Relevant flight records: {len(model_df):,}")


# ------------------------------------------------------------
# 2. Departure operations
# ------------------------------------------------------------

departures = model_df[
    model_df["ORIGIN"].astype(str).isin(MODELING_AIRPORTS)
].copy()

# Exact scheduled departure timestamp
departures["scheduled_departure"] = (
    departures["FL_DATE"]
    + pd.to_timedelta(
        departures["CRS_DEP_MIN"],
        unit="m"
    )
)

# >>> CRITICAL: convert event time to hourly bucket <<<
departures["timestamp"] = (
    departures["scheduled_departure"]
    .dt.floor("h")
)

departures["OPERATED"] = (
    departures["IS_CANCELLED"] == 0
)

departures["DELAYED_15"] = (
    departures["DEP_DELAY"]
    .fillna(0)
    .ge(15)
)

departure_hourly = (
    departures
    .groupby(["ORIGIN", "timestamp"])
    .agg(
        scheduled_departures=("ORIGIN", "size"),
        operated_departures=("OPERATED", "sum"),
        delayed_departures=("DELAYED_15", "sum"),
        avg_departure_delay=("DEP_DELAY", "mean"),
        cancellation_rate=("IS_CANCELLED", "mean"),
    )
    .reset_index()
    .rename(columns={"ORIGIN": "airport"})
)

departure_hourly["delay_rate"] = (
    departure_hourly["delayed_departures"]
    /
    departure_hourly["operated_departures"].replace(0, np.nan)
)


# ------------------------------------------------------------
# 3. Inbound operations
# ------------------------------------------------------------

arrivals = model_df[
    model_df["DEST"].astype(str).isin(MODELING_AIRPORTS)
].copy()

arrivals = arrivals[
    arrivals["CRS_ARR_MIN"].notna()
].copy()

arrivals["scheduled_arrival"] = (
    arrivals["FL_DATE"]
    + pd.to_timedelta(
        arrivals["CRS_ARR_MIN"],
        unit="m"
    )
)

# >>> CRITICAL: hourly arrival bucket <<<
arrivals["timestamp"] = (
    arrivals["scheduled_arrival"]
    .dt.floor("h")
)

arrivals["OPERATED"] = (
    arrivals["IS_CANCELLED"] == 0
)

inbound_hourly = (
    arrivals
    .groupby(["DEST", "timestamp"])
    .agg(
        inbound_flights=("DEST", "size"),
        inbound_operated=("OPERATED", "sum"),
        inbound_avg_delay=("ARR_DELAY", "mean"),
    )
    .reset_index()
    .rename(columns={"DEST": "airport"})
)


# ------------------------------------------------------------
# 4. Merge departure + inbound state
# ------------------------------------------------------------

panel = departure_hourly.merge(
    inbound_hourly,
    on=["airport", "timestamp"],
    how="outer",
)

panel = (
    panel
    .sort_values(["airport", "timestamp"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Complete the hourly time axis per airport
# ------------------------------------------------------------

complete = []

for airport, group in panel.groupby("airport"):

    group = (
        group
        .set_index("timestamp")
        .sort_index()
    )

    hourly_index = pd.date_range(
        group.index.min().floor("h"),
        group.index.max().floor("h"),
        freq="h",
    )

    group = group.reindex(hourly_index)

    group["airport"] = airport

    complete.append(group)

panel = (
    pd.concat(complete)
    .reset_index()
    .rename(columns={"index": "timestamp"})
)


# ------------------------------------------------------------
# 6. Zero-fill volume/status variables
# ------------------------------------------------------------

ZERO_COLS = [
    "scheduled_departures",
    "operated_departures",
    "delayed_departures",
    "cancellation_rate",
    "inbound_flights",
    "inbound_operated",
]

for col in ZERO_COLS:
    panel[col] = panel[col].fillna(0)


# ------------------------------------------------------------
# 7. Recalculate delay rate after zero filling
# ------------------------------------------------------------

panel["delay_rate"] = (
    panel["delayed_departures"]
    /
    panel["operated_departures"].replace(0, np.nan)
)


# ------------------------------------------------------------
# 8. Calendar features
# ------------------------------------------------------------

panel["hour"] = panel["timestamp"].dt.hour
panel["day_of_week"] = panel["timestamp"].dt.dayofweek
panel["month"] = panel["timestamp"].dt.month

panel["is_weekend"] = (
    panel["day_of_week"] >= 5
).astype("int8")

panel["is_peak"] = (
    panel["hour"].isin(
        [7, 8, 9, 16, 17, 18, 19]
    )
).astype("int8")


# ------------------------------------------------------------
# 9. Sort and validate
# ------------------------------------------------------------

panel = (
    panel
    .sort_values(["airport", "timestamp"])
    .reset_index(drop=True)
)

print()
print("=" * 65)
print("AIRPORT × HOUR PANEL CREATED")
print("=" * 65)

print(f"Rows       : {len(panel):,}")
print(f"Airports   : {panel['airport'].nunique()}")
print(
    f"Time range : "
    f"{panel['timestamp'].min()} → "
    f"{panel['timestamp'].max()}"
)

print("\nTimestamp check:")
print(
    "All timestamps exactly on the hour:",
    bool((panel["timestamp"].dt.minute == 0).all())
)

print("\nRows per airport:")
display(
    panel.groupby("airport")
    .size()
    .sort_values(ascending=False)
    .to_frame("hour_rows")
)

display(panel.head(10))

Relevant flight records: 1,320,367

AIRPORT × HOUR PANEL CREATED
Rows       : 327,163
Airports   : 8
Time range : 2019-01-01 00:00:00 → 2023-08-31 23:00:00

Timestamp check:
All timestamps exactly on the hour: True

Rows per airport:


,hour_rows
airport,
DFW,40896
EWR,40896
LAS,40896
LAX,40896
ORD,40896
MCO,40896
PHX,40896
DEN,40891


,timestamp,airport,scheduled_departures,operated_departures,delayed_departures,avg_departure_delay,cancellation_rate,delay_rate,inbound_flights,inbound_operated,inbound_avg_delay,hour,day_of_week,month,is_weekend,is_peak
0,2019-01-01 05:00:00,DEN,2.000,2.000,0.000,3.000,0.000,0.000,0.000,0.000,NaN,5,1,1,0,0
1,2019-01-01 06:00:00,DEN,0.000,0.000,0.000,NaN,0.000,NaN,1.000,1.000,10.000,6,1,1,0,0
2,2019-01-01 07:00:00,DEN,3.000,3.000,1.000,8.333,0.000,0.333,7.000,7.000,-4.286,7,1,1,0,1
3,2019-01-01 08:00:00,DEN,5.000,5.000,1.000,6.200,0.000,0.200,2.000,2.000,-9.500,8,1,1,0,1
4,2019-01-01 09:00:00,DEN,1.000,1.000,0.000,1.000,0.000,0.000,3.000,3.000,8.333,9,1,1,0,1
5,2019-01-01 10:00:00,DEN,4.000,4.000,0.000,2.750,0.000,0.000,5.000,5.000,37.200,10,1,1,0,0
6,2019-01-01 11:00:00,DEN,6.000,6.000,2.000,16.500,0.000,0.333,4.000,4.000,16.750,11,1,1,0,0
7,2019-01-01 12:00:00,DEN,3.000,3.000,1.000,21.667,0.000,0.333,2.000,2.000,12.000,12,1,1,0,0
8,2019-01-01 13:00:00,DEN,4.000,4.000,1.000,12.250,0.000,0.250,3.000,3.000,8.000,13,1,1,0,0
9,2019-01-01 14:00:00,DEN,5.000,5.000,1.000,6.000,0.000,0.200,2.000,2.000,78.000,14,1,1,0,0


In [21]:
# ============================================================
# CELL 6 — TEMPORAL FEATURES + FUTURE OUTCOME
#
# The Kaggle file is a sampled flight dataset. Some airport-hour
# buckets have no sampled flights, so raw delay-rate fields can
# be missing. For the time-series state features, an hour with
# zero operated flights is treated as zero observed delay activity.
#
# The prediction target remains based only on future hours with
# actual sampled flight activity.
# ============================================================

panel = (
    panel
    .sort_values(["airport", "timestamp"])
    .reset_index(drop=True)
)

g = panel.groupby("airport", group_keys=False)

# ------------------------------------------------------------
# 1. Stable state variables for time-series features
# ------------------------------------------------------------

# No operated departures = no observed departure-delay activity.
# Keep the distinction in operated_departures itself.
panel["delay_rate_state"] = (
    panel["delay_rate"].fillna(0)
)

panel["inbound_delay_state"] = (
    panel["inbound_avg_delay"].fillna(0)
)

# ------------------------------------------------------------
# 2. Lag features
# ------------------------------------------------------------

for lag in [1, 2, 3, 6, 24, 168]:

    panel[f"delay_lag_{lag}h"] = (
        g["delay_rate_state"].shift(lag)
    )

    if lag <= 6:
        panel[f"inbound_delay_lag_{lag}h"] = (
            g["inbound_delay_state"].shift(lag)
        )

# ------------------------------------------------------------
# 3. Rolling state
#
# shift(1) prevents the current hour from entering its own
# historical features.
# ------------------------------------------------------------

panel["delay_roll_3h"] = (
    g["delay_rate_state"]
    .transform(
        lambda x:
            x.shift(1)
             .rolling(3, min_periods=2)
             .mean()
    )
)

panel["delay_roll_6h"] = (
    g["delay_rate_state"]
    .transform(
        lambda x:
            x.shift(1)
             .rolling(6, min_periods=3)
             .mean()
    )
)

panel["delay_volatility_6h"] = (
    g["delay_rate_state"]
    .transform(
        lambda x:
            x.shift(1)
             .rolling(6, min_periods=3)
             .std()
    )
)

# ------------------------------------------------------------
# 4. Dynamics
# ------------------------------------------------------------

panel["delay_momentum_1to6h"] = (
    panel["delay_lag_1h"]
    - panel["delay_lag_6h"]
)

panel["delay_acceleration"] = (
    panel["delay_lag_1h"]
    - 2 * panel["delay_lag_2h"]
    + panel["delay_lag_3h"]
)

# ------------------------------------------------------------
# 5. Future 1h and 2h operational outcomes
# ------------------------------------------------------------

future_parts = []

for horizon in [1, 2]:

    future = panel[
        [
            "airport",
            "timestamp",
            "delay_rate",
            "operated_departures",
        ]
    ].copy()

    future["timestamp"] = (
        future["timestamp"]
        - pd.Timedelta(hours=horizon)
    )

    future = future.rename(
        columns={
            "delay_rate":
                f"future_delay_{horizon}h",
            "operated_departures":
                f"future_operated_{horizon}h",
        }
    )

    future_parts.append(future)

future = future_parts[0].merge(
    future_parts[1],
    on=["airport", "timestamp"],
    how="outer",
)

panel = panel.merge(
    future,
    on=["airport", "timestamp"],
    how="left",
)

# ------------------------------------------------------------
# 6. Future-window weighted delay rate
# ------------------------------------------------------------

panel["future_volume"] = (
    panel[
        [
            "future_operated_1h",
            "future_operated_2h",
        ]
    ]
    .fillna(0)
    .sum(axis=1)
)

future_delay_numerator = (
    panel["future_delay_1h"]
    .fillna(0)
    * panel["future_operated_1h"]
    .fillna(0)
    +
    panel["future_delay_2h"]
    .fillna(0)
    * panel["future_operated_2h"]
    .fillna(0)
)

panel["future_delay_rate"] = (
    future_delay_numerator
    / panel["future_volume"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 7. Diagnostics
# ------------------------------------------------------------

print("=" * 65)
print("TEMPORAL FEATURE CHECK")
print("=" * 65)

print(f"Panel rows: {len(panel):,}")

print("\nHistorical lag availability:")

for col in [
    "delay_lag_1h",
    "delay_lag_2h",
    "delay_lag_3h",
    "delay_lag_6h",
    "delay_lag_24h",
    "delay_lag_168h",
]:
    print(
        f"{col:<20} "
        f"{panel[col].notna().mean():.1%}"
    )

print("\nFuture-window coverage:")
print(
    f"≥1 future flight : "
    f"{(panel['future_volume'] >= 1).mean():.1%}"
)

print(
    f"≥5 future flights: "
    f"{(panel['future_volume'] >= 5).mean():.1%}"
)

display(
    panel[
        [
            "airport",
            "timestamp",
            "operated_departures",
            "delay_rate_state",
            "delay_lag_1h",
            "delay_lag_6h",
            "delay_lag_24h",
            "delay_momentum_1to6h",
            "future_volume",
            "future_delay_rate",
        ]
    ].head(12)
)

TEMPORAL FEATURE CHECK
Panel rows: 327,163

Historical lag availability:
delay_lag_1h         100.0%
delay_lag_2h         100.0%
delay_lag_3h         100.0%
delay_lag_6h         100.0%
delay_lag_24h        99.9%
delay_lag_168h       99.6%

Future-window coverage:
≥1 future flight : 75.1%
≥5 future flights: 42.0%


,airport,timestamp,operated_departures,delay_rate_state,delay_lag_1h,delay_lag_6h,delay_lag_24h,delay_momentum_1to6h,future_volume,future_delay_rate
0,DEN,2019-01-01 05:00:00,2.000,0.000,NaN,NaN,NaN,NaN,3.000,0.333
1,DEN,2019-01-01 06:00:00,0.000,0.000,0.000,NaN,NaN,NaN,8.000,0.250
2,DEN,2019-01-01 07:00:00,3.000,0.333,0.000,NaN,NaN,NaN,6.000,0.167
3,DEN,2019-01-01 08:00:00,5.000,0.200,0.333,NaN,NaN,NaN,5.000,0.000
4,DEN,2019-01-01 09:00:00,1.000,0.000,0.200,NaN,NaN,NaN,10.000,0.200
5,DEN,2019-01-01 10:00:00,4.000,0.000,0.000,NaN,NaN,NaN,9.000,0.333
6,DEN,2019-01-01 11:00:00,6.000,0.333,0.000,0.000,NaN,0.000,7.000,0.286
7,DEN,2019-01-01 12:00:00,3.000,0.333,0.333,0.000,NaN,0.333,9.000,0.222
8,DEN,2019-01-01 13:00:00,4.000,0.250,0.333,0.333,NaN,0.000,7.000,0.286
9,DEN,2019-01-01 14:00:00,5.000,0.200,0.250,0.200,NaN,0.050,2.000,0.500


In [22]:
# ============================================================
# CELL 7 — TARGET + LEAKAGE-SAFE TRAIN / VALIDATION / TEST
#
# "Disruption" means the next 2-hour operational window is
# unusually delayed for that airport.
#
# The threshold is learned ONLY from the training period.
# ============================================================

TRAIN_END = pd.Timestamp("2022-01-01")
VAL_END = pd.Timestamp("2023-01-01")

# ------------------------------------------------------------
# 1. Keep rows with useful current + future information
# ------------------------------------------------------------

model_df = panel[
    (panel["operated_departures"] >= 5) &
    (panel["future_volume"] >= 5)
].copy()

# ------------------------------------------------------------
# 2. Learn airport-specific disruption thresholds
# from TRAINING DATA ONLY
# ------------------------------------------------------------

train_target_source = model_df[
    model_df["timestamp"] < TRAIN_END
].dropna(
    subset=["future_delay_rate"]
)

thresholds = (
    train_target_source
    .groupby("airport")["future_delay_rate"]
    .quantile(0.90)
    .rename("disruption_threshold")
)

model_df["disruption_threshold"] = (
    model_df["airport"].map(thresholds)
)

# ------------------------------------------------------------
# 3. Binary disruption target
# ------------------------------------------------------------

model_df["target"] = (
    model_df["future_delay_rate"]
    >= model_df["disruption_threshold"]
).astype("int8")

# ------------------------------------------------------------
# 4. Remove rows without sufficient history
# ------------------------------------------------------------

required_history = [
    "delay_lag_1h",
    "delay_lag_2h",
    "delay_lag_3h",
    "delay_lag_6h",
]

model_df = model_df.dropna(
    subset=required_history
).copy()

# ------------------------------------------------------------
# 5. Chronological split
# ------------------------------------------------------------

train = model_df[
    model_df["timestamp"] < TRAIN_END
].copy()

val = model_df[
    (model_df["timestamp"] >= TRAIN_END)
    &
    (model_df["timestamp"] < VAL_END)
].copy()

test = model_df[
    model_df["timestamp"] >= VAL_END
].copy()

# ------------------------------------------------------------
# 6. Hard validation checks
# ------------------------------------------------------------

print("=" * 65)
print("MODEL DATASET")
print("=" * 65)

for name, data in [
    ("Train", train),
    ("Validation", val),
    ("Test", test),
]:
    print(
        f"{name:<12}"
        f"rows={len(data):>9,} | "
        f"disruption={data['target'].mean():.2%} | "
        f"range={data['timestamp'].min()} → "
        f"{data['timestamp'].max()}"
    )

if train.empty or val.empty or test.empty:
    raise RuntimeError(
        "At least one temporal split is empty. "
        "Check the target coverage diagnostics above."
    )

print("\n✓ All three splits contain data.")

MODEL DATASET
Train       rows=   23,493 | disruption=11.37% | range=2019-01-01 07:00:00 → 2021-12-31 18:00:00
Validation  rows=    8,744 | disruption=14.82% | range=2022-01-01 07:00:00 → 2022-12-31 20:00:00
Test        rows=    6,332 | disruption=19.96% | range=2023-01-01 08:00:00 → 2023-08-31 20:00:00

✓ All three splits contain data.


In [25]:
# ============================================================
# CELL 8 — FEATURE MATRIX
# All imputation statistics are learned from TRAINING only.
# ============================================================

from sklearn.impute import SimpleImputer

FEATURES = [
    "scheduled_departures",
    "operated_departures",
    "delayed_departures",
    "delay_rate_state",
    "avg_departure_delay",
    "cancellation_rate",

    "inbound_flights",
    "inbound_operated",
    "inbound_delay_state",

    "delay_lag_1h",
    "delay_lag_2h",
    "delay_lag_3h",
    "delay_lag_6h",
    "delay_lag_24h",
    "delay_lag_168h",

    "inbound_delay_lag_1h",
    "inbound_delay_lag_2h",
    "inbound_delay_lag_3h",
    "inbound_delay_lag_6h",

    "delay_roll_3h",
    "delay_roll_6h",
    "delay_volatility_6h",

    "delay_momentum_1to6h",
    "delay_acceleration",

    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_peak",
]

FEATURES = [
    c for c in FEATURES
    if c in train.columns
]

if len(FEATURES) == 0:
    raise RuntimeError("No usable model features found.")

imputer = SimpleImputer(
    strategy="median"
)

X_train = pd.DataFrame(
    imputer.fit_transform(train[FEATURES]),
    columns=FEATURES,
    index=train.index,
)

X_val = pd.DataFrame(
    imputer.transform(val[FEATURES]),
    columns=FEATURES,
    index=val.index,
)

X_test = pd.DataFrame(
    imputer.transform(test[FEATURES]),
    columns=FEATURES,
    index=test.index,
)

y_train = train["target"].to_numpy()
y_val = val["target"].to_numpy()
y_test = test["target"].to_numpy()

print("=" * 60)
print("FEATURE MATRIX READY")
print("=" * 60)

print(f"Features : {len(FEATURES)}")
print(f"Train    : {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test     : {X_test.shape}")

FEATURE MATRIX READY
Features : 29
Train    : (23493, 29)
Validation: (8744, 29)
Test     : (6332, 29)


In [26]:
# ============================================================
# CELL 8 — BASELINES
# 1) Seasonal empirical prior: airport × hour × weekday.
# 2) Logistic regression: the first true ML baseline.
# ============================================================

def metric_row(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "PR-AUC": average_precision_score(y_true, prob),
        "ROC-AUC": roc_auc_score(y_true, prob),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Brier": brier_score_loss(y_true, prob),
    }

def best_f1_threshold(y_true, prob):
    precision, recall, thresholds = precision_recall_curve(y_true, prob)
    if len(thresholds) == 0:
        return 0.5
    f1 = (
        2 * precision[:-1] * recall[:-1]
        / np.clip(precision[:-1] + recall[:-1], 1e-9, None)
    )
    return float(thresholds[np.nanargmax(f1)])

def recall_at_top_k(y_true, prob, fraction=0.10):
    k = max(1, int(len(prob) * fraction))
    top_idx = np.argsort(prob)[::-1][:k]
    return float(y_true[top_idx].sum() / max(y_true.sum(), 1))

# --- seasonal prior -------------------------------------------
train_key = (
    train["airport"].astype(str) + "_"
    + train["hour"].astype(str) + "_"
    + train["day_of_week"].astype(str)
)
val_key = (
    val["airport"].astype(str) + "_"
    + val["hour"].astype(str) + "_"
    + val["day_of_week"].astype(str)
)
test_key = (
    test["airport"].astype(str) + "_"
    + test["hour"].astype(str) + "_"
    + test["day_of_week"].astype(str)
)

prior_table = (
    pd.Series(y_train, index=train_key)
    .groupby(level=0).mean()
)
airport_prior = (
    pd.Series(y_train, index=train["airport"].astype(str))
    .groupby(level=0).mean()
)
global_prior = y_train.mean()

def prior_predict(keys, airports):
    return np.array([
        prior_table.get(
            k,
            airport_prior.get(str(a), global_prior)
        )
        for k, a in zip(keys, airports)
    ])

prior_val_prob = prior_predict(val_key, val["airport"])
prior_test_prob = prior_predict(test_key, test["airport"])

# --- logistic regression --------------------------------------
logistic = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
logistic.fit(X_train, y_train)

logistic_val_prob = logistic.predict_proba(X_val)[:, 1]
logistic_test_prob = logistic.predict_proba(X_test)[:, 1]

baseline_results = pd.DataFrame(
    [
        metric_row(y_val, prior_val_prob),
        metric_row(y_val, logistic_val_prob),
    ],
    index=["Seasonal prior", "Logistic regression"],
)

display(baseline_results.style.format("{:.3f}"))

,PR-AUC,ROC-AUC,Precision,Recall,F1,Brier
Seasonal prior,0.230,0.637,0.234,0.014,0.026,0.125
Logistic regression,0.510,0.801,0.326,0.697,0.444,0.185


In [27]:
# ============================================================
# CELL 9 — LIGHTGBM
# A fast, strong tabular baseline with controlled complexity.
# ============================================================

positive_rate = y_train.mean()
scale_pos_weight = (1 - positive_rate) / max(positive_rate, 1e-9)

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.035,
    num_leaves=31,
    min_child_samples=80,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

lgbm.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[
        early_stopping(80, verbose=False),
        log_evaluation(0),
    ],
)

lgbm_val_prob = lgbm.predict_proba(X_val)[:, 1]
lgbm_test_prob = lgbm.predict_proba(X_test)[:, 1]

print(f"Best iteration: {lgbm.best_iteration_}")
print(
    f"Validation PR-AUC: "
    f"{average_precision_score(y_val, lgbm_val_prob):.3f}"
)

Best iteration: 9
Validation PR-AUC: 0.512


In [28]:
# ============================================================
# CELL 10 — CATBOOST
# A second strong tabular learner. Use both T4s when possible;
# fall back to CPU if the GPU configuration is unavailable.
# ============================================================

catboost_params = dict(
    iterations=900,
    depth=7,
    learning_rate=0.04,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False,
    l2_leaf_reg=5,
    random_strength=0.5,
)

def build_catboost(use_gpu):
    if use_gpu:
        return CatBoostClassifier(
            **catboost_params,
            task_type="GPU",
            devices="0:1",
        )
    return CatBoostClassifier(
        **catboost_params,
        task_type="CPU",
        thread_count=-1,
    )

try:
    catboost_model = build_catboost(GPU_COUNT >= 2)
    catboost_model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=80,
        verbose=False,
    )
    print(
        "CatBoost trained on "
        + ("2 GPUs." if GPU_COUNT >= 2 else "CPU.")
    )
except Exception as gpu_error:
    print("GPU training unavailable; switching CatBoost to CPU.")
    catboost_model = build_catboost(False)
    catboost_model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=80,
        verbose=False,
    )
    print("CatBoost trained on CPU.")

catboost_val_prob = catboost_model.predict_proba(X_val)[:, 1]
catboost_test_prob = catboost_model.predict_proba(X_test)[:, 1]

print(
    f"Validation PR-AUC: "
    f"{average_precision_score(y_val, catboost_val_prob):.3f}"
)

CatBoost trained on CPU.
Validation PR-AUC: 0.534


In [29]:
# ============================================================
# CELL 11 — MODEL SELECTION + VALIDATION VISUAL
# Choose the booster on validation PR-AUC. The test set has
# still not been used for selection.
# ============================================================

val_results = pd.DataFrame(
    [
        metric_row(y_val, prior_val_prob),
        metric_row(y_val, logistic_val_prob),
        metric_row(y_val, lgbm_val_prob),
        metric_row(y_val, catboost_val_prob),
    ],
    index=[
        "Seasonal prior",
        "Logistic regression",
        "LightGBM",
        "CatBoost",
    ],
).sort_values("PR-AUC", ascending=False)

display(val_results.style.format("{:.3f}"))

BEST_BOOSTER = (
    "CatBoost"
    if val_results.loc["CatBoost", "PR-AUC"]
    >= val_results.loc["LightGBM", "PR-AUC"]
    else "LightGBM"
)

booster_val_prob = (
    catboost_val_prob
    if BEST_BOOSTER == "CatBoost"
    else lgbm_val_prob
)
booster_test_prob = (
    catboost_test_prob
    if BEST_BOOSTER == "CatBoost"
    else lgbm_test_prob
)

print(f"Selected booster: {BEST_BOOSTER}")

plot_data = (
    val_results[["PR-AUC", "ROC-AUC", "F1"]]
    .reset_index()
    .rename(columns={"index": "Model"})
    .melt(
        id_vars="Model",
        var_name="Metric",
        value_name="Score",
    )
)

fig = px.bar(
    plot_data,
    x="Model",
    y="Score",
    color="Metric",
    barmode="group",
    text="Score",
    color_discrete_sequence=[
        COLORS["blue"],
        COLORS["orange"],
        COLORS["purple"],
    ],
    title="Validation benchmark · choose before touching the test set",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    template="plotly_white",
    height=500,
    title_x=0.5,
    yaxis_range=[0, 1],
)
fig.show()

,PR-AUC,ROC-AUC,Precision,Recall,F1,Brier
CatBoost,0.534,0.816,0.713,0.267,0.389,0.096
LightGBM,0.512,0.806,0.000,0.000,0.000,0.113
Logistic regression,0.510,0.801,0.326,0.697,0.444,0.185
Seasonal prior,0.230,0.637,0.234,0.014,0.026,0.125


Selected booster: CatBoost


In [30]:
# ============================================================
# CELL 12 — FINAL STATIC TEST EVALUATION
# The operating threshold is chosen on validation only.
# Recall@Top10% measures how many disruptions we catch if only
# the top 10% of hours can be reviewed/flagged.
# ============================================================

static_threshold = best_f1_threshold(
    y_val, booster_val_prob
)

static_test_metrics = metric_row(
    y_test,
    booster_test_prob,
    threshold=static_threshold,
)
static_test_metrics["Recall@Top10%"] = recall_at_top_k(
    y_test,
    booster_test_prob,
    0.10,
)

static_result = pd.DataFrame(
    [static_test_metrics],
    index=[f"Static {BEST_BOOSTER}"],
)

print(f"Validation-selected threshold: {static_threshold:.3f}")
display(static_result.style.format("{:.3f}"))

Validation-selected threshold: 0.181


,PR-AUC,ROC-AUC,Precision,Recall,F1,Brier,Recall@Top10%
Static CatBoost,0.600,0.813,0.470,0.634,0.540,0.119,0.355


In [31]:
# ============================================================
# CELL 13 — DISCOVER LATENT OPERATING REGIMES
# Gaussian Mixture is deliberately used instead of a fragile
# extra package: it gives interpretable latent states, and the
# empirical transition matrix below restores the temporal view.
# ============================================================

REGIME_FEATURES = [
    "delay_rate",
    "avg_departure_delay",
    "scheduled_departures",
    "inbound_flights",
    "inbound_avg_delay",
    "delay_momentum_1to6h",
    "delay_acceleration",
]
REGIME_FEATURES = [
    c for c in REGIME_FEATURES if c in train.columns
]

regime_imputer = SimpleImputer(strategy="median")
regime_scaler = StandardScaler()

X_regime_train = regime_scaler.fit_transform(
    regime_imputer.fit_transform(train[REGIME_FEATURES])
)

gmm_models = {}
bic_scores = {}

for n_states in [3, 4]:
    gm = GaussianMixture(
        n_components=n_states,
        covariance_type="diag",
        n_init=5,
        random_state=RANDOM_STATE,
    )
    gm.fit(X_regime_train)
    gmm_models[n_states] = gm
    bic_scores[n_states] = gm.bic(X_regime_train)

bic_table = pd.DataFrame({
    "Latent states": list(bic_scores.keys()),
    "BIC": list(bic_scores.values()),
})

display(
    bic_table.style.format({"BIC": "{:,.0f}"})
)

N_REGIMES = min(bic_scores, key=bic_scores.get)
gmm = gmm_models[N_REGIMES]

print(f"Selected latent states: {N_REGIMES}")

,Latent states,BIC
0,3,"287,973"
1,4,"274,722"


Selected latent states: 4


In [32]:
# ============================================================
# CELL 14 — UNDERSTAND THE REGIMES
# We inspect state profiles first, then create simple descriptive
# labels. The transition matrix is computed within each airport.
# ============================================================

def assign_regimes(data):
    X = regime_scaler.transform(
        regime_imputer.transform(data[REGIME_FEATURES])
    )
    result = data[["airport", "timestamp"]].copy()
    result["regime"] = gmm.predict(X)
    return result

train_reg = assign_regimes(train)
val_reg = assign_regimes(val)
test_reg = assign_regimes(test)

train_with_reg = train.copy()
train_with_reg["regime"] = train_reg["regime"].to_numpy()

regime_profile = (
    train_with_reg.groupby("regime")[REGIME_FEATURES].mean()
)

profile_z = (
    regime_profile
    - train_with_reg[REGIME_FEATURES].mean()
) / train_with_reg[REGIME_FEATURES].std().replace(0, 1)

print("Standardized state profile:")
display(profile_z.round(2))

delay_rank = regime_profile["delay_rate"].rank()
momentum_rank = regime_profile["delay_momentum_1to6h"].rank()

REGIME_NAMES = {}
for state in regime_profile.index:
    if state == delay_rank.idxmax():
        REGIME_NAMES[state] = "High Delay"
    elif state == momentum_rank.idxmax():
        REGIME_NAMES[state] = "Rapid Acceleration"
    elif state == delay_rank.idxmin():
        REGIME_NAMES[state] = "Low Stress"
    else:
        REGIME_NAMES[state] = "Elevated"

# ----- phase-space view ---------------------------------------
phase = train_with_reg.sample(
    min(12000, len(train_with_reg)),
    random_state=RANDOM_STATE,
).copy()
phase["State"] = phase["regime"].map(REGIME_NAMES)

fig = px.scatter(
    phase,
    x="delay_rate",
    y="delay_momentum_1to6h",
    size="scheduled_departures",
    color="State",
    hover_data=["airport", "timestamp"],
    color_discrete_sequence=[
        COLORS["blue"], COLORS["cyan"],
        COLORS["orange"], COLORS["red"],
    ],
    title="Operational phase space · latent states",
)
fig.update_layout(
    template="plotly_white",
    height=520,
    title_x=0.5,
    xaxis_title="Current delay rate",
    yaxis_title="1h → 6h delay momentum",
)
fig.show()

# ----- empirical transition matrix -----------------------------
seq = train_with_reg.sort_values(
    ["airport", "timestamp"]
).copy()
seq["next_state"] = (
    seq.groupby("airport")["regime"].shift(-1)
)

transition = (
    seq.dropna(subset=["next_state"])
    .groupby(["regime", "next_state"])
    .size()
    .unstack(fill_value=0)
)
transition = transition.div(
    transition.sum(axis=1).replace(0, np.nan),
    axis=0,
)

transition.index = [
    REGIME_NAMES.get(i, f"State {i}")
    for i in transition.index
]
transition.columns = [
    REGIME_NAMES.get(i, f"State {i}")
    for i in transition.columns
]

fig = px.imshow(
    transition.fillna(0),
    text_auto=".2f",
    color_continuous_scale="Turbo",
    title="Empirical regime transition matrix",
)
fig.update_layout(
    template="plotly_white",
    height=430,
    title_x=0.5,
    xaxis_title="Next state",
    yaxis_title="Current state",
)
fig.show()

Standardized state profile:


,delay_rate,avg_departure_delay,scheduled_departures,inbound_flights,inbound_avg_delay,delay_momentum_1to6h,delay_acceleration
regime,,,,,,,
0,-0.940,-0.620,-0.210,-0.060,-0.230,-0.110,-0.010
1,-0.140,-0.280,0.250,-0.030,-0.250,-0.100,-0.000
2,0.670,0.360,0.020,0.070,-0.030,0.090,0.010
3,1.670,1.980,-0.000,0.060,1.630,0.410,-0.010


In [33]:
# ============================================================
# CELL 15 — REGIME-AWARE MODEL
# Add the latent state as one-hot features to the strongest
# static booster. No other part of the ML setup changes.
# ============================================================

train_r = train.copy()
val_r = val.copy()
test_r = test.copy()

train_r["regime"] = train_reg["regime"].to_numpy()
val_r["regime"] = val_reg["regime"].to_numpy()
test_r["regime"] = test_reg["regime"].to_numpy()

X_train_r = X_train.copy()
X_val_r = X_val.copy()
X_test_r = X_test.copy()

for state in sorted(
    set(train_r["regime"])
    | set(val_r["regime"])
    | set(test_r["regime"])
):
    X_train_r[f"regime_{state}"] = (
        train_r["regime"].eq(state).astype("int8").to_numpy()
    )
    X_val_r[f"regime_{state}"] = (
        val_r["regime"].eq(state).astype("int8").to_numpy()
    )
    X_test_r[f"regime_{state}"] = (
        test_r["regime"].eq(state).astype("int8").to_numpy()
    )

if BEST_BOOSTER == "CatBoost":
    try:
        regime_model = build_catboost(GPU_COUNT >= 2)
        regime_model.fit(
            X_train_r, y_train,
            eval_set=(X_val_r, y_val),
            early_stopping_rounds=80,
            verbose=False,
        )
    except Exception:
        regime_model = build_catboost(False)
        regime_model.fit(
            X_train_r, y_train,
            eval_set=(X_val_r, y_val),
            early_stopping_rounds=80,
            verbose=False,
        )
else:
    regime_model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.035,
        num_leaves=31,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.2,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )
    regime_model.fit(
        X_train_r,
        y_train,
        eval_set=[(X_val_r, y_val)],
        eval_metric="auc",
        callbacks=[
            early_stopping(80, verbose=False),
            log_evaluation(0),
        ],
    )

regime_val_prob = regime_model.predict_proba(X_val_r)[:, 1]
regime_test_prob = regime_model.predict_proba(X_test_r)[:, 1]

regime_threshold = best_f1_threshold(
    y_val, regime_val_prob
)

regime_metrics = metric_row(
    y_test,
    regime_test_prob,
    threshold=regime_threshold,
)
regime_metrics["Recall@Top10%"] = recall_at_top_k(
    y_test, regime_test_prob, 0.10
)

comparison = pd.DataFrame(
    [static_test_metrics, regime_metrics],
    index=[
        f"Static {BEST_BOOSTER}",
        f"Regime-aware {BEST_BOOSTER}",
    ],
)

display(comparison.style.format("{:.3f}"))

,PR-AUC,ROC-AUC,Precision,Recall,F1,Brier,Recall@Top10%
Static CatBoost,0.600,0.813,0.470,0.634,0.540,0.119,0.355
Regime-aware CatBoost,0.600,0.811,0.492,0.622,0.549,0.119,0.354


In [34]:
# ============================================================
# CELL 16 — FINAL EVALUATION: OVERALL + BY REGIME
# The main question is not simply "did AUC move?" but:
# "where, if anywhere, did regime information help?"
# ============================================================

plot_cmp = (
    comparison[
        ["PR-AUC", "ROC-AUC", "Recall", "F1", "Recall@Top10%"]
    ]
    .reset_index()
    .rename(columns={"index": "Model"})
    .melt(
        id_vars="Model",
        var_name="Metric",
        value_name="Score",
    )
)

fig = px.bar(
    plot_cmp,
    x="Metric",
    y="Score",
    color="Model",
    barmode="group",
    text="Score",
    color_discrete_sequence=[
        COLORS["gray"], COLORS["orange"]
    ],
    title="Does regime information improve the untouched test set?",
)
fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
)
fig.update_layout(
    template="plotly_white",
    height=500,
    title_x=0.5,
    yaxis_range=[0, 1],
)
fig.show()

# --- by regime ------------------------------------------------
eval_test = test_r[
    ["airport", "timestamp", "regime"]
].copy()
eval_test["actual"] = y_test
eval_test["static_prob"] = booster_test_prob
eval_test["regime_prob"] = regime_test_prob

rows = []
for state in sorted(eval_test["regime"].unique()):
    mask = eval_test["regime"].eq(state)

    if mask.sum() < 100:
        continue
    if eval_test.loc[mask, "actual"].nunique() < 2:
        continue

    rows.append({
        "Regime": REGIME_NAMES.get(state, f"State {state}"),
        "Hours": int(mask.sum()),
        "Static PR-AUC": average_precision_score(
            eval_test.loc[mask, "actual"],
            eval_test.loc[mask, "static_prob"],
        ),
        "Regime PR-AUC": average_precision_score(
            eval_test.loc[mask, "actual"],
            eval_test.loc[mask, "regime_prob"],
        ),
    })

regime_performance = pd.DataFrame(rows)

display(
    regime_performance.style.format({
        "Hours": "{:,.0f}",
        "Static PR-AUC": "{:.3f}",
        "Regime PR-AUC": "{:.3f}",
    })
)

if len(regime_performance):
    rp = regime_performance.melt(
        id_vars=["Regime", "Hours"],
        value_vars=["Static PR-AUC", "Regime PR-AUC"],
        var_name="Model",
        value_name="PR-AUC",
    )

    fig = px.bar(
        rp,
        x="Regime",
        y="PR-AUC",
        color="Model",
        barmode="group",
        text="PR-AUC",
        color_discrete_sequence=[
            COLORS["gray"], COLORS["orange"]
        ],
        title="Where does regime awareness help?",
    )
    fig.update_traces(
        texttemplate="%{text:.3f}",
        textposition="outside",
    )
    fig.update_layout(
        template="plotly_white",
        height=480,
        title_x=0.5,
    )
    fig.show()

,Regime,Hours,Static PR-AUC,Regime PR-AUC
0,Low Stress,"1,588",0.158,0.148
1,Elevated,"1,336",0.231,0.225
2,Elevated,"2,456",0.424,0.419
3,High Delay,952,0.839,0.840


In [36]:
# ============================================================
# CELL 17 — SHAP EXPLANATION
# Explain the regime-aware model on a small test sample.
# ============================================================

sample_n = min(3000, len(X_test_r))
X_shap = X_test_r.sample(
    sample_n,
    random_state=RANDOM_STATE,
)

explainer = shap.TreeExplainer(regime_model)
shap_values = explainer.shap_values(X_shap)

# SHAP returns different shapes across versions/models.
if isinstance(shap_values, list):
    shap_values = shap_values[-1]
if np.ndim(shap_values) == 3:
    shap_values = shap_values[:, :, -1]

importance = (
    pd.Series(
        np.abs(shap_values).mean(axis=0),
        index=X_shap.columns,
    )
    .sort_values(ascending=True)
    .tail(12)
)

fig = px.bar(
    importance,
    orientation="h",
    color=importance.values,
    color_continuous_scale="Turbo",
    title="What drives near-term disruption risk?",
    labels={
        "value": "Mean |SHAP value|",
        "index": "Feature",
    },
)
fig.update_layout(
    template="plotly_white",
    height=560,
    title_x=0.5,
    showlegend=False,
)
fig.show()

display(
    importance.sort_values(ascending=False)
    .rename("mean_abs_shap")
    .to_frame()
)

,mean_abs_shap
inbound_delay_state,0.379
avg_departure_delay,0.262
inbound_delay_lag_1h,0.199
delay_rate_state,0.190
delay_roll_6h,0.170
inbound_delay_lag_2h,0.130
delay_roll_3h,0.112
delay_lag_24h,0.108
inbound_delay_lag_3h,0.102
month,0.099


In [37]:
# ============================================================
# CELL 18 — FINAL EVENT-LEVEL CASE STUDY
# One airport, one short window, three aligned signals:
# model risk, observed delay, and operating regime.
# ============================================================

case_eval = test_r[
    ["airport", "timestamp", "regime"]
].copy()
case_eval["actual_delay_rate"] = test["delay_rate"].to_numpy()
case_eval["static_prob"] = booster_test_prob
case_eval["regime_prob"] = regime_test_prob
case_eval["regime_label"] = case_eval["regime"].map(REGIME_NAMES)

case_airport = (
    case_eval["airport"]
    .value_counts()
    .index[0]
)

case = (
    case_eval[case_eval["airport"] == case_airport]
    .sort_values("timestamp")
    .head(24 * 14)
    .copy()
)

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    vertical_spacing=0.07,
    subplot_titles=(
        "Predicted disruption probability",
        "Observed delay rate + latent regime",
    ),
)

fig.add_trace(
    go.Scatter(
        x=case["timestamp"],
        y=case["static_prob"],
        mode="lines",
        name="Static model",
        line=dict(color=COLORS["gray"], width=2),
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=case["timestamp"],
        y=case["regime_prob"],
        mode="lines",
        name="Regime-aware model",
        line=dict(color=COLORS["orange"], width=3),
    ),
    row=1, col=1,
)

fig.add_hline(
    y=0.5,
    line_dash="dash",
    line_color=COLORS["red"],
    annotation_text="50% alert threshold",
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=case["timestamp"],
        y=case["actual_delay_rate"] * 100,
        mode="lines",
        name="Observed delay rate",
        line=dict(color=COLORS["blue"], width=2),
    ),
    row=2, col=1,
)

fig.add_trace(
    go.Scatter(
        x=case["timestamp"],
        y=case["regime"],
        mode="markers",
        name="Latent regime",
        marker=dict(
            size=8,
            color=case["regime"],
            colorscale="Turbo",
            showscale=False,
        ),
        text=case["regime_label"],
        hovertemplate=(
            "Regime: %{text}<br>%{x}<extra></extra>"
        ),
    ),
    row=2, col=1,
)

fig.update_yaxes(
    title_text="Probability",
    range=[0, 1],
    row=1, col=1,
)
fig.update_yaxes(
    title_text="Delay rate (%) / state",
    row=2, col=1,
)
fig.update_xaxes(
    title_text="Time",
    row=2, col=1,
)
fig.update_layout(
    template="plotly_white",
    height=700,
    title=f"{case_airport} — A disruption episode in context",
    title_x=0.5,
    hovermode="x unified",
)
fig.show()

delta_pr = (
    comparison.iloc[1]["PR-AUC"]
    - comparison.iloc[0]["PR-AUC"]
)

print(f"Regime-aware PR-AUC change: {delta_pr:+.3f}")
print(
    "The final conclusion comes from the untouched test period; "
    "no outcome was selected in advance."
)

Regime-aware PR-AUC change: -0.001
The final conclusion comes from the untouched test period; no outcome was selected in advance.


### Literature context

This notebook is an applied synthesis rather than a new-method paper. Recent flight-delay literature continues to emphasize nonlinear ensemble models, temporal/spatial dependence, and explainability; recent aviation work also studies regime-switching/disruption-aware forecasting.

- Flight-delay ML systematic review (2026): https://doi.org/10.1016/j.asoc.2026.115761
- Regime-switching aviation forecasting (2026): https://doi.org/10.1016/j.jairtraman.2026.103055
- Tabular foundation-model benchmark (Nature): https://doi.org/10.1038/s41586-024-08328-6

# Final takeaway

The project is intentionally honest and focused:

**flight events → operational time series → strong tabular ML → latent regimes → regime-aware prediction → explainability**

The key result is not “we used a fancy model.” It is whether **recognizing changing operating states adds useful predictive information when the airport's behavior changes**.